# Politics dataset for the Polymarket project

This notebook builds `politics.csv`, one row per market, with the columns:

`slug, category, volume, close_date, price_7d, price_3d, price_1d, outcome`

Everything downloaded is kept in the `raw_politics` folder, so a second run reuses it instead of downloading again.

## Decision log

Filters and outcomes:

* Downloaded the 4,000 highest-volume closed events tagged Politics
* Dropped 2,346 events with more than one market
* Dropped 81 events also tagged Crypto or Tech
* Dropped 6 markets with no scheduled deadline (`endDate`)
* Dropped 481 markets whose deadline was outside 1 January 2025 to 12 November 2026
* All markets passed the 10k volume filter
* Dropped 11 markets that did not resolve cleanly Yes or No (cancelled, settled 50/50, or not a Yes/No question)
* Took the outcome from the final prices
* Measured the 7, 3 and 1 day points back from the scheduled deadline (`endDate`), not the actual closing time (`closedTime`). When a market closes depends on how it resolves: a "by date" market that resolves Yes closes soon after the event happens, so counting back from the close would pick the time points using the answer. The deadline is fixed when the market is created
* Dropped 221 markets that had already closed at one of the three time points. Most of them resolved Yes, so the sample leans slightly towards questions still undecided near their deadline
* Kept only the highest-volume market of each repeated question, dropping 167. Repeated questions are the same question asked for different dates (e.g. "Will X happen by ..."), grouped by Polymarket's `seriesSlug`; their outcomes move together
* Used hourly prices, and took the last price at or before each time point, as long as it was at most 24 hours old
* Dropped 105 markets missing one of the three prices
* Kept the 500 highest-volume markets that were left

In [123]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import pandas as pd
import requests

## Settings

The two Polymarket APIs:
- Gamma holds events, markets, tags and final outcomes
- CLOB holds the price histories.

In [124]:
GAMMA_ADDRESS = "https://gamma-api.polymarket.com"
CLOB_ADDRESS = "https://clob.polymarket.com"

The Politics tag, and the tags that outrank it

In [125]:
POLITICS_TAG_ID = 2
HIGHER_PRIORITY_TAG_IDS = ["21", "1401"]

How much to download and what to measure:

- `EVENTS_PER_PAGE`: events per request (100 is the most the server allows).
- `PAGES_PER_ROUND`: how many more pages to fetch each time we run short.
- `MARKETS_WANTED`: rows in the final table.
- `DAYS_BEFORE_DEADLINE`: the three time points before a market's scheduled deadline.

In [126]:
EVENTS_PER_PAGE = 100
PAGES_PER_ROUND = 10
MARKETS_WANTED = 500
DAYS_BEFORE_DEADLINE = [7, 3, 1]

**Design choice.** A price only counts if it is at most this many hours old at the time point. This also drops markets that were open for less than 7 days.

In [127]:
MAX_PRICE_AGE_HOURS = 24

**Design choice.** A market only counts if at least this many dollars were traded on it. A price on a market where only a few hundred dollars ever traded is one or two people's guess, not a market forecast.

In [128]:
MIN_VOLUME = 10000

Date range for the deadline (`endDate`), written as `"YYYY-MM-DD"`

In [129]:
EARLIEST_DEADLINE = "2025-01-01"
LATEST_DEADLINE = "2026-11-12"

Where the raw downloads and the result are saved, the exact column order of the result, and the pause between two requests (to be polite to the server).

In [130]:
RAW_EVENTS_FOLDER = "raw_politics/events"
RAW_HISTORIES_FOLDER = "raw_politics/histories_deadline"
OUTPUT_FILE = "politics.csv"
COLUMNS = ["slug", "category", "volume", "close_date",
           "price_7d", "price_3d", "price_1d", "outcome"]
PAUSE_SECONDS = 0.2

## Small helpers

`get_json` fetches an address and returns the reply as Python lists and dictionaries. If the server is busy (status 429 or 500 and above) it waits and tries again, up to 5 times. Other failures stop with an error.

In [131]:
def get_json(address, settings):
    for attempt in range(5):
        try:
            reply = requests.get(address, params=settings, timeout=30)
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
            continue
        if reply.status_code == 200:
            return reply.json()
        if reply.status_code == 429 or reply.status_code >= 500:
            time.sleep(2 * (attempt + 1))
            continue
        raise RuntimeError(f"{address} replied {reply.status_code}: {reply.text[:200]}")
    raise RuntimeError(f"{address} kept failing after 5 tries")

`read_file` and `write_file` load and save data as JSON files (both APIs return in JSON format)

`write_file` creates the folder first if it does not exist

In [132]:
def read_file(file_name):
    with open(file_name) as file:
        return json.load(file)


def write_file(data, file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    with open(file_name, "w") as file:
        json.dump(data, file)

`text_to_time` turns a time written as text into a real time in UTC in two formats: `2024-11-06T20:40:10Z` and `2024-11-06 15:17:41+00`.

In [133]:
def text_to_time(text):
    text = text.strip().replace(" ", "T")
    if text.endswith("Z"):
        text = text[:-1] + "+00:00"
    elif text.endswith("+00"):
        text = text + ":00"
    return datetime.fromisoformat(text).astimezone(timezone.utc)

The fields `outcomes`, `outcomePrices` and `clobTokenIds` arrive as text that looks like a list, e.g. `'["Yes", "No"]'`

`text_to_list` turns such text into a real list.

In [134]:
def text_to_list(text):
    if not text:
        return []
    return json.loads(text)

## Step 1: download the events

`fetch_events_page` asks for one page of closed politics events, highest volume first

`cursor` is where the previous page ended (`None` for the first page)

It returns the page as the server sent it: `{"events": [...], "next_cursor": ...}`.

In [135]:
def fetch_events_page(cursor):
    settings = {
        "tag_id": POLITICS_TAG_ID,
        "closed": "true",
        "order": "volume",
        "ascending": "false",
        "limit": EVENTS_PER_PAGE,
    }
    if cursor:
        settings["after_cursor"] = cursor
    return get_json(GAMMA_ADDRESS + "/events/keyset", settings)

`load_events` returns the events from the first `number_of_pages` pages. Each page is saved to its own file and only downloaded if the file is missing. When the server has no more pages it stops early.

Returns `(events, more_pages_exist)`.

In [136]:
def load_events(number_of_pages):
    events = []
    cursor = None
    for page_number in range(1, number_of_pages + 1):
        file_name = f"{RAW_EVENTS_FOLDER}/page_{page_number:04d}.json"
        if os.path.exists(file_name):
            page = read_file(file_name)
        else:
            page = fetch_events_page(cursor)
            write_file(page, file_name)
            time.sleep(PAUSE_SECONDS)
        events.extend(page.get("events", []))
        cursor = page.get("next_cursor")
        if not cursor:
            return events, False
    return events, True

## Step 2: build the market list

`market_outcome` works out how a market resolved, from its final prices. It finds where "Yes" sits in the outcomes list and uses that same position for the final price and the token ID.

Returns `(outcome, yes_token_id)`.

The outcome is 1 if Yes won, 0 if No won, and `None` if the market is unusable (not a Yes/No market, cancelled, or settled 50/50).

In [137]:
def market_outcome(market):
    outcomes = text_to_list(market.get("outcomes"))
    final_prices = text_to_list(market.get("outcomePrices"))
    token_ids = text_to_list(market.get("clobTokenIds"))

    if sorted(outcomes) != ["No", "Yes"]:
        return None, None
    if len(final_prices) != 2 or len(token_ids) != 2:
        return None, None

    yes_position = outcomes.index("Yes")
    yes_final_price = float(final_prices[yes_position])
    if yes_final_price == 1:
        return 1, token_ids[yes_position]
    if yes_final_price == 0:
        return 0, token_ids[yes_position]
    return None, None

`reason_to_drop` applies the filters from decision log

It returns `None` if the event passes, otherwise the reason it is dropped

A market needs a scheduled deadline (`endDate`), because the time points are measured back from it, and the deadline must fall inside the date range. It also needs at least `MIN_VOLUME` dollars traded

In [138]:
def reason_to_drop(event):
    markets = event.get("markets") or []
    if len(markets) != 1:
        return "event has more than one market"

    tag_ids = [str(tag.get("id")) for tag in event.get("tags") or []]
    for higher_tag_id in HIGHER_PRIORITY_TAG_IDS:
        if higher_tag_id in tag_ids:
            return "also tagged Crypto or Tech"

    market = markets[0]
    if not market.get("closedTime"):
        return "no closing time"

    if not market.get("endDate"):
        return "no deadline"

    deadline_date = text_to_time(market["endDate"]).strftime("%Y-%m-%d")
    if EARLIEST_DEADLINE is not None and deadline_date < EARLIEST_DEADLINE:
        return "deadline outside the date range"
    if LATEST_DEADLINE is not None and deadline_date > LATEST_DEADLINE:
        return "deadline outside the date range"

    if float(market.get("volume") or 0) < MIN_VOLUME:
        return "volume under the minimum"

    outcome, yes_token_id = market_outcome(market)
    if outcome is None:
        return "did not resolve cleanly Yes or No"
    return None

`build_market_list` turns the events into one record per usable market (slug, volume, closing time, deadline, question group, Yes token ID, outcome), sorted from highest volume to lowest

It also counts how many events were dropped for each reason

The question group is Polymarket's own `seriesSlug`: repeated questions asked for different dates (e.g. "Will the Iranian regime fall by ...") share one. A market with no series is its own group, named after its event

In [139]:
def build_market_list(events):
    markets = []
    drop_counts = {}
    slugs_seen = set()

    for event in events:
        reason = reason_to_drop(event)
        if reason is None and event["markets"][0]["slug"] in slugs_seen:
            reason = "duplicate"
        if reason is not None:
            drop_counts[reason] = drop_counts.get(reason, 0) + 1
            continue

        market = event["markets"][0]
        outcome, yes_token_id = market_outcome(market)
        slugs_seen.add(market["slug"])
        markets.append({
            "slug": market["slug"],
            "volume": float(market.get("volume") or 0),
            "closed_time": text_to_time(market["closedTime"]),
            "deadline": text_to_time(market["endDate"]),
            "question_group": event.get("seriesSlug") or event["slug"],
            "yes_token_id": yes_token_id,
            "outcome": outcome,
        })

    markets.sort(key=lambda market: market["volume"], reverse=True)
    return markets, drop_counts

## Step 3: download the price histories

`fetch_price_history` gets the hourly Yes prices (`fidelity` 60 = one point per 60 minutes) for the stretch we need: from just before the 7-day point up to the deadline

Each point is `{"t": seconds since 1970, "p": price}`

In [140]:
def fetch_price_history(market):
    longest_look_back = timedelta(days=max(DAYS_BEFORE_DEADLINE), hours=MAX_PRICE_AGE_HOURS + 1)
    settings = {
        "market": market["yes_token_id"],
        "startTs": int((market["deadline"] - longest_look_back).timestamp()),
        "endTs": int(market["deadline"].timestamp()),
        "fidelity": 60,
    }
    reply = get_json(CLOB_ADDRESS + "/prices-history", settings)
    return reply.get("history", [])

`load_history` returns the market's price history, downloaded only if its file is missing.

In [141]:
def load_history(market):
    file_name = f"{RAW_HISTORIES_FOLDER}/{market['slug']}.json"
    if os.path.exists(file_name):
        return read_file(file_name)
    history = fetch_price_history(market)
    write_file(history, file_name)
    time.sleep(PAUSE_SECONDS)
    return history

## Step 4: build the table

`price_before_deadline` finds the last price at or before (deadline − `days_before`)

It returns `None` if there is no price there, or if the last one is older than `MAX_PRICE_AGE_HOURS`

In [142]:
def price_before_deadline(history, deadline, days_before):
    target_seconds = (deadline - timedelta(days=days_before)).timestamp()
    oldest_allowed = target_seconds - MAX_PRICE_AGE_HOURS * 3600

    best_point = None
    for point in history:
        if oldest_allowed <= point["t"] <= target_seconds:
            if best_point is None or point["t"] > best_point["t"]:
                best_point = point
    if best_point is None:
        return None
    return best_point["p"]

`build_rows` goes down the markets from highest volume and makes a row for each one that has all three prices

A market that had already closed at one of the time points is skipped before its history is downloaded: it has no price there to judge

Only the highest-volume market of each question group gets a row: once a group has a row, its other markets are skipped, because their outcomes move together

It stops once `MARKETS_WANTED` rows are made, and prints progress every 50 markets

Returns `(rows, missing_price_count, closed_early_count, repeated_question_count)`

In [143]:
def build_rows(markets):
    rows = []
    missing_price_count = 0
    closed_early_count = 0
    repeated_question_count = 0
    groups_used = set()

    for position, market in enumerate(markets, start=1):
        if len(rows) == MARKETS_WANTED:
            break
        if market["question_group"] in groups_used:
            repeated_question_count += 1
            continue
        time_points = [market["deadline"] - timedelta(days=days) for days in DAYS_BEFORE_DEADLINE]
        if market["closed_time"] <= max(time_points):
            closed_early_count += 1
            continue
        history = load_history(market)
        prices = [price_before_deadline(history, market["deadline"], days)
                  for days in DAYS_BEFORE_DEADLINE]
        if None in prices:
            missing_price_count += 1
            continue
        groups_used.add(market["question_group"])
        rows.append({
            "slug": market["slug"],
            "category": "Politics",
            "volume": market["volume"],
            "close_date": market["closed_time"].strftime("%Y-%m-%d"),
            "price_7d": prices[0],
            "price_3d": prices[1],
            "price_1d": prices[2],
            "outcome": market["outcome"],
        })
        if position % 50 == 0:
            print(f"  checked {position} markets, {len(rows)} rows so far")

    return rows, missing_price_count, closed_early_count, repeated_question_count

`check_table` stops with a error if the table breaks the plan's format:
- wrong columns or order
- blank cells
- a repeated slug
- a price outside 0 to 1
- an outcome that is not 0 or 1
- a category other than Politics

In [144]:
def check_table(table):
    assert list(table.columns) == COLUMNS, "wrong columns or wrong order"
    assert not table.isnull().any().any(), "there are blank cells"
    assert not table["slug"].duplicated().any(), "a slug appears twice"
    for column in ["price_7d", "price_3d", "price_1d"]:
        assert table[column].between(0, 1).all(), f"{column} has a value outside 0 to 1"
    assert table["outcome"].isin([0, 1]).all(), "outcome must be 0 or 1"
    assert (table["category"] == "Politics").all(), "category must be Politics"

## Run everything

`main` runs the four stages in order. If fewer than `MARKETS_WANTED` rows come out and the server still has more events, it fetches `PAGES_PER_ROUND` more pages and tries again. Then it checks the table, saves it and prints a summary

In [145]:
def main():
    number_of_pages = PAGES_PER_ROUND

    while True:
        events, more_pages_exist = load_events(number_of_pages)
        markets, drop_counts = build_market_list(events)
        print(f"{number_of_pages} pages: {len(events)} events, "
              f"{len(markets)} markets pass the filters")

        rows, missing_price_count, closed_early_count, repeated_question_count = build_rows(markets)
        if len(rows) >= MARKETS_WANTED or not more_pages_exist:
            break
        number_of_pages += PAGES_PER_ROUND

    table = pd.DataFrame(rows, columns=COLUMNS)
    check_table(table)
    table.to_csv(OUTPUT_FILE, index=False)

    print()
    print("Events downloaded:        ", len(events))
    for reason, count in sorted(drop_counts.items()):
        print(f"  dropped, {reason}: {count}")
    print("Markets passing filters:  ", len(markets))
    print("Dropped, repeated question:", repeated_question_count)
    print("Dropped, already closed at a time point:", closed_early_count)
    print("Dropped, missing a price: ", missing_price_count)
    print("Rows saved:               ", len(table), "to", OUTPUT_FILE)
    if len(table) < MARKETS_WANTED:
        print("WARNING: ran out of events before reaching", MARKETS_WANTED, "rows")

Run main

In [146]:
main()

10 pages: 1000 events, 191 markets pass the filters
  checked 50 markets, 22 rows so far
  checked 150 markets, 61 rows so far
20 pages: 2000 events, 464 markets pass the filters
  checked 50 markets, 22 rows so far
  checked 150 markets, 61 rows so far
  checked 300 markets, 128 rows so far
  checked 350 markets, 156 rows so far
  checked 450 markets, 213 rows so far
30 pages: 3000 events, 775 markets pass the filters
  checked 50 markets, 22 rows so far
  checked 150 markets, 61 rows so far
  checked 300 markets, 128 rows so far
  checked 350 markets, 156 rows so far
  checked 450 markets, 213 rows so far
  checked 550 markets, 271 rows so far
40 pages: 4000 events, 1075 markets pass the filters
  checked 50 markets, 22 rows so far
  checked 150 markets, 61 rows so far
  checked 300 markets, 128 rows so far
  checked 350 markets, 156 rows so far
  checked 450 markets, 213 rows so far
  checked 550 markets, 271 rows so far
  checked 800 markets, 395 rows so far
  checked 900 markets, 

## Result

In [147]:
pd.read_csv(OUTPUT_FILE)

,slug,category,volume,close_date,price_7d,price_3d,price_1d,outcome
0,will-there-be-another-us-government-shutdown-b...,Politics,1.572966e+08,2026-02-04,0.0850,0.7850,0.4950,1
1,khamenei-out-as-supreme-leader-of-iran-by-febr...,Politics,1.311150e+08,2026-03-04,0.0400,0.0210,0.0135,1
2,russia-x-ukraine-ceasefire-in-2025,Politics,7.375299e+07,2026-01-01,0.0095,0.0120,0.0020,0
3,will-the-iranian-regime-fall-by-june-30,Politics,6.593194e+07,2026-07-01,0.0025,0.0015,0.0005,0
4,will-biden-complete-his-term-as-president,Politics,6.455217e+07,2025-01-20,0.9915,0.9925,0.9935,1
...,...,...,...,...,...,...,...,...
495,will-netanyahu-be-pardoned-in-2025,Politics,1.274755e+05,2026-01-01,0.0130,0.0130,0.0145,0
496,will-trump-pardon-maduro-by-december-31,Politics,1.262619e+05,2026-01-01,0.0055,0.0030,0.0025,0
497,us-operation-to-capture-maduro-in-2025,Politics,1.224189e+05,2026-01-01,0.0205,0.0245,0.0075,0
498,trump-imposes-tariff-on-pharmaceuticals-before...,Politics,1.219137e+05,2025-05-01,0.1800,0.0850,0.0850,0
